In [ ]:
"""
Monte Carlo (MC) Dropout on MNIST, A to Z.

Steps:
  1. Load data (MNIST for training/testing, FashionMNIST as out-of-distribution)
  2. Define the model (with dropout)
  3. Train normally
  4. Standard evaluation (dropout OFF)
  5. MC Dropout evaluation (dropout ON, T stochastic passes)
  6. Uncertainty: predictive entropy, mutual information
  7. Accuracy vs. uncertainty (reject the most uncertain samples)
  8. In-distribution vs. out-of-distribution uncertainty
  9. Plots
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# 0. Setup
# --------------------------------------------------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
T = 50  # number of MC dropout samples

# --------------------------------------------------------------------------
# 1. Data
# --------------------------------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_set = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST("./data", train=False, download=True, transform=transform)
# Out-of-distribution data (the model has never seen clothing items)
ood_set = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1000, shuffle=False)
ood_loader = DataLoader(ood_set, batch_size=1000, shuffle=False)

# --------------------------------------------------------------------------
# 2. Model
# --------------------------------------------------------------------------
model = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(p=0.2),
    nn.Linear(1 * 28 * 28, 100),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(100, 10),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


# --------------------------------------------------------------------------
# 3. Training
# --------------------------------------------------------------------------
def train_one_epoch(model, loader):
    model.train()  # dropout ON during training
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n


for epoch in range(1, EPOCHS + 1):
    loss, acc = train_one_epoch(model, train_loader)
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {loss:.4f} | train acc {acc:.4f}")


# --------------------------------------------------------------------------
# 4. Standard evaluation (dropout OFF)
# --------------------------------------------------------------------------
@torch.no_grad()
def evaluate_standard(model, loader):
    model.eval()  # dropout OFF
    correct, n = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        n += x.size(0)
    return correct / n


std_acc = evaluate_standard(model, test_loader)
print(f"\nStandard test accuracy (dropout off): {std_acc:.4f}")


# --------------------------------------------------------------------------
# 5. MC Dropout
# --------------------------------------------------------------------------
def enable_dropout(model):
    """Eval mode for everything, but keep Dropout layers in train mode."""
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


@torch.no_grad()
def mc_dropout_predict(model, x, T=50):
    """
    Run T stochastic forward passes.
    Returns softmax probabilities of shape (T, N, num_classes).
    """
    enable_dropout(model)
    probs = torch.stack([F.softmax(model(x), dim=1) for _ in range(T)])
    return probs


# --------------------------------------------------------------------------
# 6. Uncertainty measures
# --------------------------------------------------------------------------
def uncertainty_from_probs(probs, eps=1e-12):
    """
    probs: (T, N, C)
    Returns:
      mean_probs  (N, C) : predictive distribution
      pred_entropy (N,)  : total uncertainty  H[ E_t p_t ]
      exp_entropy  (N,)  : aleatoric part     E_t H[p_t]
      mutual_info  (N,)  : epistemic part     pred_entropy - exp_entropy
    """
    mean_probs = probs.mean(0)
    pred_entropy = -(mean_probs * (mean_probs + eps).log()).sum(-1)
    exp_entropy = -(probs * (probs + eps).log()).sum(-1).mean(0)
    mutual_info = pred_entropy - exp_entropy
    return mean_probs, pred_entropy, exp_entropy, mutual_info


@torch.no_grad()
def mc_evaluate(model, loader, T=50):
    """Run MC dropout over a full loader and collect everything on CPU."""
    all_mean, all_ent, all_mi, all_y = [], [], [], []
    for x, y in loader:
        x = x.to(device)
        probs = mc_dropout_predict(model, x, T)
        mean_probs, ent, _, mi = uncertainty_from_probs(probs)
        all_mean.append(mean_probs.cpu())
        all_ent.append(ent.cpu())
        all_mi.append(mi.cpu())
        all_y.append(y)
    return (torch.cat(all_mean), torch.cat(all_ent),
            torch.cat(all_mi), torch.cat(all_y))


mean_probs, ent, mi, y_true = mc_evaluate(model, test_loader, T)
mc_pred = mean_probs.argmax(1)
correct_mask = (mc_pred == y_true)
mc_acc = correct_mask.float().mean().item()
print(f"MC dropout test accuracy (T={T}):      {mc_acc:.4f}")

print(f"\nMean predictive entropy | correct: {ent[correct_mask].mean():.4f}"
      f" | wrong: {ent[~correct_mask].mean():.4f}")
print(f"Mean mutual information | correct: {mi[correct_mask].mean():.4f}"
      f" | wrong: {mi[~correct_mask].mean():.4f}")

# --------------------------------------------------------------------------
# 7. Accuracy vs. uncertainty (keep only the most confident samples)
# --------------------------------------------------------------------------
order = ent.argsort()  # most certain first
fractions = np.linspace(0.1, 1.0, 10)
retained_acc = []
for f in fractions:
    k = int(f * len(order))
    idx = order[:k]
    retained_acc.append(correct_mask[idx].float().mean().item())

print("\nAccuracy when keeping only the most certain predictions:")
for f, a in zip(fractions, retained_acc):
    print(f"  keep {int(f * 100):3d}% -> accuracy {a:.4f}")

# --------------------------------------------------------------------------
# 8. Out-of-distribution (FashionMNIST) uncertainty
# --------------------------------------------------------------------------
_, ood_ent, ood_mi, _ = mc_evaluate(model, ood_loader, T)
print(f"\nMean predictive entropy | MNIST (in-dist): {ent.mean():.4f}"
      f" | FashionMNIST (OOD): {ood_ent.mean():.4f}")
print(f"Mean mutual information | MNIST (in-dist): {mi.mean():.4f}"
      f" | FashionMNIST (OOD): {ood_mi.mean():.4f}")

# --------------------------------------------------------------------------
# 9. Plots
# --------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# (a) entropy: correct vs wrong
axes[0].hist(ent[correct_mask].numpy(), bins=50, alpha=0.6, density=True, label="correct")
axes[0].hist(ent[~correct_mask].numpy(), bins=50, alpha=0.6, density=True, label="wrong")
axes[0].set_title("Predictive entropy: correct vs wrong")
axes[0].set_xlabel("Entropy")
axes[0].legend()

# (b) in-distribution vs OOD
axes[1].hist(ent.numpy(), bins=50, alpha=0.6, density=True, label="MNIST")
axes[1].hist(ood_ent.numpy(), bins=50, alpha=0.6, density=True, label="FashionMNIST (OOD)")
axes[1].set_title("Predictive entropy: in-dist vs OOD")
axes[1].set_xlabel("Entropy")
axes[1].legend()

# (c) accuracy vs fraction retained
axes[2].plot(fractions * 100, retained_acc, marker="o")
axes[2].set_title("Accuracy vs fraction of data kept")
axes[2].set_xlabel("% of most certain samples kept")
axes[2].set_ylabel("Accuracy")
axes[2].grid(True)

plt.tight_layout()
plt.savefig("mc_dropout_uncertainty.png", dpi=150)
plt.show()

# (d) most uncertain test images
top_idx = ent.argsort(descending=True)[:10]
fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
for ax, i in zip(axes.ravel(), top_idx):
    img, label = test_set[i.item()]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"true {label} | pred {mc_pred[i].item()}\nH={ent[i]:.2f}")
    ax.axis("off")
plt.suptitle("Most uncertain MNIST test images")
plt.tight_layout()
plt.savefig("mc_dropout_most_uncertain.png", dpi=150)
plt.show()


# (e) predictive distribution for a single image
def show_single_prediction(index=0, T=100):
    img, label = test_set[index]
    x = img.unsqueeze(0).to(device)
    probs = mc_dropout_predict(model, x, T)[:, 0, :].cpu()  # (T, C)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img.squeeze(), cmap="gray")
    axes[0].set_title(f"True label: {label}")
    axes[0].axis("off")

    axes[1].bar(range(10), probs.mean(0).numpy(), yerr=probs.std(0).numpy(), capsize=3)
    axes[1].set_xticks(range(10))
    axes[1].set_xlabel("Class")
    axes[1].set_ylabel("Probability (mean ± std over MC samples)")
    axes[1].set_title(f"MC dropout prediction (T={T})")
    plt.tight_layout()
    plt.show()


show_single_prediction(index=0)

# --------------------------------------------------------------------------
# Save model
# --------------------------------------------------------------------------
torch.save(model.state_dict(), "mc_dropout_mnist.pth")
print("\nSaved model to mc_dropout_mnist.pth")

Using device: cpu


100.0%
100.0%
100.0%
100.0%
100.0%
100.0%
100.0%
100.0%
